# ABSA tiếng Việt: stacking encoder học có giám sát trên CPU

Mục tiêu người dùng: ABSA-F1 công khai 0,96. **Chưa đạt mục tiêu**.
Người dùng đã gửi ảnh điểm công khai **0,8840** cho file ứng viên, nhóm KPW2-NHOM3.
public_score_report.json lưu nguồn chứng cứ và hash file tương ứng. Đây là số
người dùng cung cấp, không phải notebook tự đăng nhập trang chấm.
Kết quả hiện đo bằng macro F1 của 14 nhãn khen/chê trên 938 review validation,
không lấy Accuracy hoặc tuning F1 thay cho điểm chính.

Notebook mặc định nạp checkpoint và các xác suất out-of-fit đã lưu khi train;
fit lại meta bằng tuning và suy luận test thực sự bằng các encoder đó. Đặt
RETRAIN_BASES=True để huấn luyện từ đầu; hướng dẫn chi tiết trong
HUONG_DAN_STACKING.md. Lịch sử train cũ không được ghi thành train mới.

Không dùng API/LLM để tạo nhãn test, không đưa review test vào prompt, không
hiển thị text test. **Bộ 4 encoder của notebook chính không dùng review/nhãn
tổng hợp.** Thử nghiệm thêm 1.400 mẫu tổng hợp được báo cáo riêng và chưa được
chọn vào bộ này. LLM hỗ trợ code và diễn giải lỗi train. Gói nộp không chứa
train.csv/test_blind.csv gốc.

In [1]:
import os,sys,json,platform,time,hashlib,subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
os.environ.setdefault('HF_HOME',str(Path('private_cache/hf').resolve()))
import torch
torch.set_num_threads(2)
import stacked_absa,neural_evaluation,pipeline
import score_absa as scorer
from phobert_absa import split_indices,predictions,to_frame
DATA=Path(os.environ.get('ABSA_DATA_DIR','..')).resolve()
RETRAIN_BASES=False
RECOMPUTE_BASE_VALIDATION=False
RUN_BASELINE_TRAINING=False
choices=[
    [Path('model_bundle')/p for p in ['cached_phobert','balanced_phobert','aspect_pair','visobert']],
    [Path(p) for p in ['cached_phobert_results','balanced_phobert_results','aspect_pair_adapter','visobert_adapter']],
    [Path(p) for p in ['cached_phobert_results','balanced_phobert_results','aspect_pair_results','visobert_results']]]
MODEL_DIRS=next((ps for ps in choices if all((p/'config.json').exists() and
    ((p/'adapter.pt').exists() or (p/'model.pt').exists()) for p in ps)),choices[-1])
if RETRAIN_BASES:
    MODEL_DIRS=[Path('retrained_models')/p for p in ['global','balanced','aspect_pair','visobert']]
STACK=Path('stacked_notebook_results')
OUTPUT=Path('submission.csv')
REFERENCE=OUTPUT.read_bytes() if OUTPUT.exists() else None
print('Python',platform.python_version(),'Torch',torch.__version__,'GPU used: False')
print('Train SHA256',stacked_absa.file_hash(DATA/'train.csv'))

/workspace/work/deep_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python 3.12.14 Torch 2.6.0+cpu GPU used: False
Train SHA256 665b581d11c5cb59eb5b3e6e730660fdd28567c64392f2e288e375dd082bd95e


## Chia dữ liệu và huấn luyện

Seed2026, group theo review chuẩn hóa, 2.812 fit / 938 tuning / 938 validation.
Chia review trước khi mở rộng thành 7 cặp. Encoder không học tuning hay validation;
meta học tuning, chọn cấu hình/ngưỡng bằng dự đoán 5-fold out-of-fold trong tuning.
Vì vậy stacking dùng 3.750 review góp phần học, encoder đơn chỉ dùng 2.812.
Validation đã được xem trong các vòng phát triển trước; không phải một phép
xác nhận mới hoàn toàn. Không đổi split để tạo điểm cao hơn.

In [2]:
def run_script(name,*args):
    subprocess.run([sys.executable,name,*map(str,args)],check=True)
def ready(p):
    return (p/'config.json').exists() and ((p/'adapter.pt').exists() or (p/'model.pt').exists())
if RETRAIN_BASES or not all(ready(p) for p in MODEL_DIRS):
    if not RETRAIN_BASES:
        raise FileNotFoundError('Thiếu checkpoint: tải model bundle hoặc đặt RETRAIN_BASES=True.')
    g,b,p,v=MODEL_DIRS
    run_script('finetune_cpu_cached.py','--data-dir',DATA,'--out-dir',g,
        '--cache','private_cache/phobert_frozen8_160.pt','--epochs',12,'--layers',4,
        '--threads',2,'--batch-size',8,'--accumulation',2,'--lr',.00003)
    run_script('finetune_cpu_cached.py','--data-dir',DATA,'--out-dir',b,
        '--cache','private_cache/phobert_frozen8_160.pt','--epochs',4,'--layers',4,
        '--threads',2,'--batch-size',8,'--accumulation',2,'--lr',.00001,'--init-dir',g,'--balanced-loss')
    run_script('aspect_pair_absa.py','--data-dir',DATA,'--out-dir',p,
        '--cache','private_cache/aspect_pair.pt','--epochs',6,'--patience',2,'--layers',4,
        '--threads',2,'--batch-size',16,'--accumulation',1,'--lr',.00003,
        '--max-length',160,'--stride',32,'--negatives',2)
    run_script('phobert_absa.py','train','--data-dir',DATA,'--out-dir',v,
        '--model-name','uitnlp/visobert','--revision','196a62afad9cbe4f52a54aabad828b13f0eec59a',
        '--fast-tokenizer','--no-segmentation','--unfreeze-layers',12,'--epochs',8,
        '--patience',2,'--batch-size',8,'--accumulation',2,'--cpu-threads',2,
        '--lr',.00002,'--max-length',160,'--stride',32,'--no-refit','--skip-test')
else:
    print('Nạp 4 encoder đã huấn luyện; chưa train encoder mới trong lần chạy này.')
for p in MODEL_DIRS:
    cfg=json.loads((p/'config.json').read_text())
    assert not cfg.get('production_refit',False) and not cfg.get('refit_scope')
    assert cfg['seed']==2026 and cfg['columns']==scorer.COLS
    print(p.name,cfg['model_name'],cfg.get('architecture','joint_21'))
    display(pd.read_csv(p/'history.csv'))
baseline_dir=Path('notebook_baseline_results')
if RUN_BASELINE_TRAINING or not (baseline_dir/'metrics.json').exists():
    with threadpool_limits(limits=1):pipeline.run(DATA,baseline_dir)
else:print('Baseline: dùng số đo và lịch sử đã lưu từ lần huấn luyện trước.')

Nạp 4 encoder đã huấn luyện; chưa train encoder mới trong lần chạy này.
cached_phobert vinai/phobert-base-v2 joint_21


    epoch      loss   tune_F1     seconds
0       1  0.683245  0.638451  269.494450
1       2  0.379576  0.744523  300.265382
2       3  0.261316  0.776592  279.920256
3       4  0.201629  0.809350  348.143952
4       5  0.155141  0.829874  214.729147
5       6  0.119213  0.845473  245.965094
6       7  0.093636  0.854246  286.308792
7       8  0.075287  0.859278  254.797176
8       9  0.064456  0.860285  278.799047
9      10  0.058443  0.863825  340.726705
10     11  0.055300  0.862679  289.350864
11     12  0.053816  0.863256  317.501794

balanced_phobert vinai/phobert-base-v2 joint_21


   epoch      loss   tune_F1     seconds
0      1  0.043974  0.864378  249.562762
1      2  0.035367  0.861081  370.829936
2      3  0.030135  0.864936  240.799372
3      4  0.027375  0.866394  321.759009

aspect_pair vinai/phobert-base-v2 aspect_pair


   epoch      loss   tune_F1     seconds
0      1  0.467742  0.768524  643.335154
1      2  0.216457  0.855317  644.041889
2      3  0.141836  0.870883  644.405994
3      4  0.095002  0.873330  645.131410
4      5  0.067201  0.871714  645.450112
5      6  0.052511  0.872405  643.186563

visobert uitnlp/visobert joint_21


   epoch      loss   tune_F1     seconds
0      1  0.618334  0.672342  723.536454
1      2  0.325215  0.749827  730.757967
2      3  0.221263  0.785337  727.806241
3      4  0.153991  0.814548  726.246720
4      5  0.108871  0.825146  725.675101
5      6  0.085908  0.833761  730.217524
6      7  0.073075  0.833385  725.968027
7      8  0.069132  0.835186  724.872096

Baseline: dùng số đo và lịch sử đã lưu từ lần huấn luyện trước.


## Tái lập meta và báo cáo validation

Mặc định dùng các xác suất tuning/validation được lưu từ checkpoint đã chọn.
Chương trình kiểm tra ID/thứ tự/split và SHA256 cấu hình + trọng số. Đặt
RECOMPUTE_BASE_VALIDATION=True để chạy lại forward của từng encoder cho cả hai
tập. Các file score lưu không chứa văn bản review. Meta không nhìn nhãn validation
khi chọn C/feature/ngưỡng; validation chỉ được dùng để báo cáo sau lựa chọn.

In [3]:
train=pd.read_csv(DATA/'train.csv')
fi,tu,va=split_indices(train,2026)
y=train[scorer.COLS].to_numpy(int)
assert (len(fi),len(tu),len(va))==(2812,938,938)
if RECOMPUTE_BASE_VALIDATION:
    import gc
    for p in MODEL_DIRS:
        cfg,m,tok=neural_evaluation.open_model(p,2)
        for part,indices in [('tune',tu),('validation',va)]:
            s=neural_evaluation.probabilities_for(cfg,m,tok,train.text.iloc[indices],8)
            np.savez_compressed(p/(part+'_scores.npz'),scores=s,ids=train.id.iloc[indices].to_numpy(dtype=str))
        del m,tok
        gc.collect()
with threadpool_limits(limits=1):
    report=stacked_absa.fit(DATA,MODEL_DIRS,STACK,2026)
baseline_all=json.loads((baseline_dir/'metrics.json').read_text())
comparison=pd.read_csv(baseline_dir/'comparison.csv')
baseline_name=comparison.sort_values('tune_F1',ascending=False).iloc[0]['model']
rows=[]
fields=['ABSA_F1','precision_TB','recall_TB','accuracy_TB','kappa_TB','mcc_TB','bal_acc_TB']
rows.append({'method':'TF-IDF '+baseline_name,**{k:baseline_all[baseline_name][k] for k in fields}})
for p in MODEL_DIRS:
    r=json.loads((p/'metrics.json').read_text())
    rows.append({'method':p.name,**{k:r[k] for k in fields}})
rows.append({'method':'stacking meta 5-fold',**{k:report[k] for k in fields}})
display(pd.DataFrame(rows))
print('ABSA-F1 CI95%',report['ABSA_F1_CI95'])
print('Target 0.96 on validation reached:',report['ABSA_F1']>=.96)
display(pd.DataFrame([{'aspect':a,'polarity':p,**d[p]} for a,d in report['theo_khia_canh'].items() for p in ['khen','che']]))
print('Nhãn yếu nhất:',report['F1_nhan_thap_nhat'])

nhan_vien_nhac meta OOF F1 0.9696 own_aspect 1.0


nhan_vien_khen meta OOF F1 0.9749 own_aspect 1.0


nhan_vien_che meta OOF F1 0.856 own_aspect 0.01


sach_se_nhac meta OOF F1 0.9767 own_aspect 0.01


sach_se_khen meta OOF F1 0.9918 own_aspect 0.01


sach_se_che meta OOF F1 0.9333 own_aspect 0.01


... (16 outputs left out)

                 method   ABSA_F1  precision_TB  recall_TB  accuracy_TB  \
0          TF-IDF nbsvm  0.756126      0.767699   0.750061     0.920804   
1        cached_phobert  0.853783      0.865604   0.846344     0.954386   
2      balanced_phobert  0.857719      0.858666   0.861732     0.955605   
3           aspect_pair  0.866615      0.862025   0.874193     0.959108   
4              visobert  0.832149      0.835902   0.834356     0.947685   
5  stacking meta 5-fold  0.880738      0.897846   0.865350     0.963448   

   kappa_TB    mcc_TB  bal_acc_TB  
0  0.702699  0.704407    0.847783  
1  0.823662  0.824989    0.908038  
2  0.828528  0.829988    0.916243  
3  0.839705  0.840615    0.923676  
4  0.798075  0.799980    0.901905  
5  0.856582  0.857145    0.922521  

ABSA-F1 CI95% [0.8655377220188366, 0.8938021691510577]
Target 0.96 on validation reached: False


       aspect polarity        f1  precision    recall  accuracy     kappa  \
0   nhan_vien     khen  0.963636   0.963636  0.963636  0.961620  0.923004   
1   nhan_vien      che  0.847926   0.859813  0.836364  0.964819  0.828039   
2     sach_se     khen  0.980519   0.986928  0.974194  0.987207  0.970996   
3     sach_se      che  0.836879   0.907692  0.776316  0.975480  0.823710   
4      vi_tri     khen  0.941558   0.963455  0.920635  0.961620  0.913009   
5      vi_tri      che  0.807339   0.800000  0.814815  0.977612  0.795456   
6       phong     khen  0.935897   0.951140  0.921136  0.957356  0.903961   
7       phong      che  0.870229   0.912000  0.832117  0.927505  0.820079   
8     gia_tri     khen  0.937729   0.948148  0.927536  0.981876  0.927125   
9     gia_tri      che  0.776699   0.833333  0.727273  0.975480  0.763790   
10    an_uong     khen  0.935211   0.943182  0.927374  0.975480  0.920091   
11    an_uong      che  0.897059   0.910448  0.884058  0.970149  0.879607   

Nhãn yếu nhất: {'nhan': 'tien_nghi_che', 'f1': 0.7761194029850746}


## Thời gian và phần cứng

Thời gian từng epoch gồm train và tuning. Tổng thời gian của các job khác nhau
không phải thời gian chờ khi chạy song song. Run cached đầu có lần resume;
total_seconds của process resume không phải tổng huấn luyện từ đầu. Cache
PhoBERT chung ban đầu mất khoảng 433,4 giây; xem hardware.json của model gốc.

In [4]:
hardware={'python':platform.python_version(),'os':platform.platform(),'torch':str(torch.__version__),
    'gpu_used':False,'cpu':next((s.split(':',1)[1].strip() for s in Path('/proc/cpuinfo').read_text().splitlines() if s.startswith('model name')),''),
    'ram':next(s for s in Path('/proc/meminfo').read_text().splitlines() if s.startswith('MemTotal:')),
    'cpu_quota':Path('/sys/fs/cgroup/cpu.max').read_text().strip(),
    'container_memory_limit_bytes':Path('/sys/fs/cgroup/memory.max').read_text().strip()}
print(json.dumps(hardware,ensure_ascii=False,indent=2))
cost=[]
for p in MODEL_DIRS:
    h=pd.read_csv(p/'history.csv')
    cost.append({'model':p.name,'epochs_recorded':len(h),'sum_epoch_seconds':float(h.seconds.sum())})
    print(p.name,'run_info:',(p/'run_info.json').read_text())
display(pd.DataFrame(cost))

{
  "python": "3.12.14",
  "os": "Linux-6.18.57-x86_64-with-glibc2.41",
  "torch": "2.6.0+cpu",
  "gpu_used": false,
  "cpu": "INTEL(R) XEON(R) PLATINUM 8573C",
  "ram": "MemTotal:       10207480 kB",
  "cpu_quota": "200000 100000",
  "container_memory_limit_bytes": "8589934592"
}
cached_phobert run_info: {
  "best_tune_F1": 0.8638252661669253,
  "best_epoch": 10,
  "validation_F1": 0.8537834976124399,
  "total_seconds": 2310.200228193,
  "cache_key": {
    "model": "vinai/phobert-base-v2",
    "revision": "86cd7fd4c148980922ac11a2cf5e257f2ba639e1",
    "frozen_depth": 8,
    "max_length": 160,
    "stride": 32,
    "train_sha256": "665b581d11c5cb59eb5b3e6e730660fdd28567c64392f2e288e375dd082bd95e"
  },
  "torch": "2.6.0+cpu",
  "threads": 2,
  "gpu": false,
  "fit_rows": 2812,
  "tune_rows": 938,
  "validation_rows": 938
}
balanced_phobert run_info: {
  "best_tune_F1": 0.8663940797168517,
  "best_epoch": 4,
  "validation_F1": 0.8577191997789887,
  "total_seconds": 1268.3613042300003,
 

              model  epochs_recorded  sum_epoch_seconds
0    cached_phobert               12        3426.002658
1  balanced_phobert                4        1182.951078
2       aspect_pair                6        3865.551123
3          visobert                8        5815.080130

## Năm lỗi trên review TRAIN được giữ làm validation

Cell này chỉ in văn bản từ train.csv, tuyệt đối không đọc text test. Bản nộp riêng
cho giảng viên giữ output; bản công khai phải xóa output nguyên văn của cell này.
Nhận xét nhằm mô tả lỗi, không vá dự đoán theo ID hoặc sửa nhãn gốc.

Bản công khai xóa output nguyên văn; bản nộp riêng giữ đủ 5 ví dụ.

In [ ]:
pred=pd.read_csv(STACK/'validation_prediction.csv')
assert np.array_equal(pred.id.to_numpy(),train.id.iloc[va].to_numpy())
vp=pred[scorer.COLS].to_numpy(int)
comments={
 'train_0012':'Bỏ sót lời chê gián tiếp khi bữa sáng buffet không đúng kỳ vọng; nhiều lời khen trong đoạn dài có thể lấn át.',
 'train_0013':'Mùi thơm là dấu hiệu sạch sẽ gián tiếp; mô hình bỏ sót khi không có từ sạch/vệ sinh rõ ràng.',
 'train_0039':'Dự đoán nhắc phòng thừa từ cụm tiện nghi chung. Lỗi nhắc này không làm giảm điểm chính 14 nhãn.',
 'train_0065':'Cảnh báo mất nón ở xe gián tiếp nói tới an toàn/tiện nghi đỗ xe, mô hình bỏ sót khía cạnh và lời chê.',
 'train_0080':'Chấp nhận được trong tầm giá là chê nhẹ theo quy ước, bị hiểu thành khen giá trị; đồng thời lan truyền lời chê sang tiện nghi.'}
for i in np.flatnonzero((vp!=y[va]).any(1))[:5]:
    row=train.iloc[va[i]]
    wrong=np.flatnonzero(vp[i]!=y[va[i]])
    print('ID',row.id)
    print(row.text)
    print({scorer.COLS[j]:{'truth':int(y[va[i],j]),'prediction':int(vp[i,j])} for j in wrong})
    print(comments.get(row.id,'Kiểm tra phạm vi khía cạnh, lời chê nhẹ và dấu hiệu gián tiếp; không chỉnh riêng dự đoán của mẫu.'))
    print()

## Dự đoán test bằng checkpoint và kiểm tra tái lập

Chỉ các encoder đã học nhận review test cục bộ. Không in text test, không gọi
API dự đoán, không tải CSV lên đâu. Predictor kiểm tra đúng byte hash checkpoint
và cấu hình, đúng thứ tự model/cột. Lần chạy sau dùng cùng checkpoint để đối chiếu
CSV. Điểm public/private cần một lần nộp được xác thực, không suy ra từ validation.

In [6]:
started=time.perf_counter()
with threadpool_limits(limits=1):
    stacked_absa.predict(STACK,DATA/'test_blind.csv',OUTPUT,threads=2,batch_size=8,
        score_cache_dir='private_compact_predictions')
elapsed=time.perf_counter()-started
assert scorer.kiem_tra(OUTPUT,DATA/'test_blind.csv')==0
if REFERENCE is not None:
    assert REFERENCE==OUTPUT.read_bytes()
    print('CSV khớp chính xác tham chiếu có trước khi chạy notebook.')
print('Prediction seconds this run (may reuse hash-checked local scores):',elapsed)
cost=Path('inference_cost.json')
if cost.exists():print('Recorded cold inference:',cost.read_text())
print('Submission SHA256:',stacked_absa.file_hash(OUTPUT))
evidence=Path('public_score_report.json')
if evidence.exists():
    public=json.loads(evidence.read_text())
    if public['associated_candidate_sha256']==stacked_absa.file_hash(OUTPUT):
        print('Điểm công khai do người dùng cung cấp qua ảnh:',public['ABSA_F1'],'; nhóm',public['group'])
    else:print('CSV này khác phiên bản trong chứng cứ công khai; không gán điểm cũ cho file mới.')
print('Mục tiêu 0.96 chưa được xác nhận.')

Local encoder scores ready: 1 / 4


Local encoder scores ready: 2 / 4


Local encoder scores ready: 3 / 4


Local encoder scores ready: 4 / 4


File hợp lệ, có thể nộp.
Submission SHA256: 6b0bd799cb3270ce87d3f2cbc6828fc50a08b261299b1dd0ceacc76db8af735d


File hợp lệ, có thể nộp.
CSV khớp chính xác tham chiếu có trước khi chạy notebook.
Prediction seconds this run (may reuse hash-checked local scores): 0.7219139759999962
Recorded cold inference: {
  "cold_prediction_seconds": 952.497788977,
  "cpu_threads": 2,
  "reviews": 732,
  "encoders": 4,
  "concurrent_training": true,
  "submission_sha256": "6b0bd799cb3270ce87d3f2cbc6828fc50a08b261299b1dd0ceacc76db8af735d"
}
Submission SHA256: 6b0bd799cb3270ce87d3f2cbc6828fc50a08b261299b1dd0ceacc76db8af735d
Điểm công khai do người dùng cung cấp qua ảnh: 0.884 ; nhóm KPW2-NHOM3
Mục tiêu 0.96 chưa được xác nhận.


## Thử nghiệm dữ liệu train bổ sung, chưa được chọn

Bộ chính ở trên không dùng dữ liệu tổng hợp. Một lượt ablation riêng warm-start
token-attention, thêm 1.400 review tổng hợp với trọng số 0,4 trên fit gốc, giữ
nguyên tuning/validation: model đơn tăng validation 0,865875 → 0,868441; bộ
kết hợp có thêm encoder này đạt 0,880435, chưa vượt 0,880738 của bộ chính.

Công cụ: ChatGPT/Codex soạn bộ mẫu cố định theo đặc tả dưới đây; Python ghép
theo seed2026, không gọi API sinh dữ liệu riêng. Không dùng hoặc viết lại review
test. du_lieu_them.csv và du_lieu_them.generation.json phải được giữ trong gói
báo cáo thử nghiệm để kiểm tra. Không dùng dữ liệu này cho bản đã chấm 0,8840.

In [7]:
from generate_training_augmentation import AUTHORING_SPEC
print('Đặc tả dùng để soạn bộ mẫu train mới:')
print(AUTHORING_SPEC)
generation=next((p for p in [Path('du_lieu_them.generation.json'),Path('augmentation_trial/du_lieu_them.generation.json')] if p.exists()),None)
if generation:
    g=json.loads(generation.read_text())
    print('Công cụ:',g['tool'])
    print('Dòng tổng hợp:',g['rows'],'; seed:',g['seed'],'; SHA256:',g['augmentation_sha256'])
ablation=next((p for p in [Path('experiments/augmentation_ablation.csv'),Path('augmentation_results/augmentation_ablation.csv')] if p.exists()),None)
if ablation:display(pd.read_csv(ablation))

Đặc tả dùng để soạn bộ mẫu train mới:
Soạn các mẫu câu tiếng Việt mới để bổ sung dữ liệu HUẤN LUYỆN
ABSA khách sạn. Chỉ dựa trên 7 định nghĩa khía cạnh trong đề, không dùng văn bản
test hoặc viết lại các review gốc. Mỗi mẫu chỉ đánh giá một khía cạnh rõ ràng.
Khen/chê chung cả khách sạn không thuộc khía cạnh nào. Bình thường, tạm được,
đề nghị cải thiện là chê. Tiếng ồn từ đường phố thuộc vị trí; cách âm giữa phòng,
thiết bị trong phòng thuộc phòng; WiFi, gym, thang máy, hồ bơi chung thuộc tiện
nghi. Quán ăn xung quanh thuộc vị trí, nhà hàng trong khách sạn thuộc ăn uống.
Ghép các mẫu theo seed, cho phép cùng một khía cạnh vừa khen vừa chê; nhãn của
review ghép là OR nhãn của các câu. Lưu đầy đủ du_lieu_them.csv và mô tả công cụ.
Công cụ: ChatGPT/Codex assistant authored fixed templates; seeded Python combines them; no separate generation API
Dòng tổng hợp: 1400 ; seed: 2026 ; SHA256: 79dc8ad6655d8c20f86175dd8e57b68bb94588a479612e306f5b37b7d4bbc063


                                 method   tune_F1  validation_F1
0        without synthetic augmentation  0.870349       0.865875
1  with weighted synthetic augmentation  0.875941       0.868441